In [ ]:
#------------------------------------------------------------------------------------------------------#
#
# Code:      CC01_CCAR_C01_AgenticAI_model_scoring_01.ipynb
#
# Objective: Step C01: Use Agentic AI to score vintages of 201605, 201606, 201607, ..., 201801.
#                      It is the basic level-1 Agentic AI, which at most has some Agentic AI concept only.
#
#            Jingru Chen
#            2026-03-22
#
#----------------------------------------------------------------------------------------------------#

In [ ]:
### From Linear Scripting to Autonomous Agency
### (1) A standard script is Passive. It requires us to tell it exactly what to do at every step.
### (2) An Agent is Task-Oriented; we give it a goal (a date range), and it manages the internal complexity of fulfilling that goal.

###   Level	         Type	           Financial Risk Example (CCAR)
###   Level-1	  Deterministic	       A Python script that runs a model on a schedule. No "thinking."

# Step 0: Upload libraries

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from pydantic import BaseModel, Field

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

run_date="2026-03-22"

start = datetime.now( ZoneInfo("America/New_York"))

print( start.strftime("%Y-%m-%d %H:%M:%S %Z"))     # 2026-03-17 17:34:58 EDT
print( start.strftime("%Y-%m-%d %I:%M:%S %p %Z"))  # 2026-03-17 05:34:58 PM EDT

2026-03-22 22:04:16 EDT
2026-03-22 10:04:16 PM EDT


In [ ]:
myout= "/content/sample_data"

In [ ]:
pwd

'/content'

In [ ]:
cd /content/sample_data/

/content/sample_data


In [ ]:
ls -ltr

total 84036
-rwxr-xr-x 1 root root      962 Jan  1  2000 README.md*
-rwxr-xr-x 1 root root     1697 Jan  1  2000 anscombe.json*
-rw-r--r-- 1 root root  1706430 Mar 17 17:58 california_housing_train.csv
-rw-r--r-- 1 root root   301141 Mar 17 17:58 california_housing_test.csv
-rw-r--r-- 1 root root 36523880 Mar 17 17:58 mnist_train_small.csv
-rw-r--r-- 1 root root 18289443 Mar 17 17:58 mnist_test.csv
-rw-r--r-- 1 root root     2209 Mar 22 21:11 ccar_pd_model_2026-03-22.pkl
-rw-r--r-- 1 root root 29210174 Mar 22 21:11 CCAR_Mortgage_data_for_model_DEV_20260319_01.csv


In [ ]:
x_list=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate', 'loan_term_months',
        'delta_Unemployment1',
       'delta_Mortgage_rate1', 'delta_House_Price_Index__Level1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_House_Price_Index__Level3', 'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12', 'delta_Mortgage_rate12',
       'delta_House_Price_Index__Level12', 'delta_Unemployment24',
       'delta_Mortgage_rate24', 'delta_House_Price_Index__Level24']

x_list_v2=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate',
        'delta_Unemployment1',
       'delta_Unemployment3',
       'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12',
       'delta_Unemployment24' ]

# y_list= ['flag_default']

pd_model = 'ccar_pd_model_2026-03-22.pkl'
pd_input_file= "/CCAR_Mortgage_data_for_model_DEV_20260319_01.csv"
# pd_vintage = 201604
pd_x_list = x_list_v2
pd_y = 'flag_default'
pd_threshold = 0.5


In [ ]:
def my_mortgage_PD_scoring( my_model, my_data, my_vintage, my_x_list, my_y, my_threshold ):

  # Step A:Load the model from your disk/storage
  pd_model = joblib.load( my_model )

  # Step B: Upload data and select a targeted vintage
  df_mortgage= pd.read_csv( myout + my_data )
  if 'Unnamed: 0' in df_mortgage.columns:
        df_mortgage = df_mortgage.drop(columns=['Unnamed: 0'])

  print( "\n------------- Step B-1: Column List of df_mortgage ---------\n", df_mortgage.shape )

  # Step C: Selectne targeted vintage for PD model scoring - CRITICAL: Cast to string to ensure matching
  df_mortgage_vintage = df_mortgage.loc[df_mortgage['report_yrmo'].astype(str) == str(my_vintage)].reset_index(drop=True)

  if df_mortgage_vintage.empty:
    print(f"!!! Error: No data found for vintage {my_vintage}")
    return None

  print( "\n------------- Step C-1: Column List of df_mortgage --------:\n", df_mortgage_vintage.report_yrmo.value_counts() )

  print( "------------- Step C-2: Summary Table of df_mortgage ---------\n", pd.crosstab( index= df_mortgage_vintage['report_yrmo'],
                                              columns= df_mortgage_vintage[my_y], margins=True))

  # Step D: Set up X vs y from the selected vintage file
  X = df_mortgage_vintage[ my_x_list ]
  y = df_mortgage_vintage[ my_y ]

  # Step E: Scoring & Confusion matrix
  ### E-1. Generate Predictions
  ### Get probabilities instead of hard predictions
  y_proba = pd_model.predict_proba(X)[:, 1]

  ### E-2: Set a custom threshold based on your portfolio's average default rate
  custom_threshold = my_threshold
  y_pred_new = (y_proba >= custom_threshold).astype(int)

  ### E-3 Add results back to the dataframe for the Agent to use
  df_mortgage_vintage['PD_Score'] = y_proba
  df_mortgage_vintage['Predicted_Class'] = y_pred_new

  ### E-4. Print the Comprehensive Classification Report
  print("\n\n------- E-1: Classification Report -------")
  print(classification_report(y, y_pred_new ))

  ### E-5. Display the Confusion Matrix
  print("------ E-2 Confusion Matrix ------")
  print(confusion_matrix(y, y_pred_new ))

  return df_mortgage_vintage # Returning the dataframe is essential for the Agent

# Step 1: Defining the Agent's "Tool"

First, let's wrap my existing function of B05 into a "Tool" that an AI Agent can understand.
Here we use Pydantic to define the schema so the Agent knows exactly what inputs are required.

In [ ]:
from pydantic import BaseModel, Field

# Defines a Pydantic model. This is the "Schema" the Agent uses to understand scoring_tool() function. This acts as the instruction manual for the AI Agent.
class ScoringInput(BaseModel):
    vintage: str = Field(description="The mortgage vintage in YYYYMM format, e.g., '201604'")
    # This is the most important part for Agentic AI (like LangChain or AutoGen).
    # The LLM (like GPT-4 or Claude) actually reads this description to understand what it needs to find in a user's transcript or database.

def scoring_tool(vintage: str):
    print(f'\n=============== Agent acting on vintage: {vintage} ===============')
    try:
        # 1. Run the scoring function
        df_scored = my_mortgage_PD_scoring(pd_model, pd_input_file, vintage, pd_x_list, pd_y, pd_threshold)

        # 2. Guard Clause
        if df_scored is None or df_scored.empty:
            return {"vintage": vintage, "status": "Skipped"}

        # 3. Extract y and y_pred
        y_true = df_scored[pd_y]
        y_pred = df_scored['Predicted_Class']

        # 4. Calculate Confusion Matrix components
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        # 5. Calculate Metrics
        # return a flat dictionary for easy DataFrame conversion
        return {
            "vintage": vintage,
            "status": "Success",
            "count": len(df_scored),
            "avg_pd": df_scored['PD_Score'].mean(),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "TP": tp, "FN": fn, "TN": tn, "FP": fp
        }

    except Exception as e:
        return {"vintage": vintage, "status": f"Error: {str(e)}"}


# Step 2: The Agentic Loop (Using a "ReAct" Pattern)

Instead of a simple for loop, an Agent uses a ReAct (Reason + Act) logic.

It looks at the goal, decides which vintage to score next, and checks if it finished the range.

In [ ]:
def run_ccar_agent(start_vintage, end_vintage):
    vintages = pd.date_range(start=pd.to_datetime(start_vintage, format='%Y%m'),
                             end=pd.to_datetime(end_vintage, format='%Y%m'),
                             freq='MS').strftime('%Y%m').tolist()

    results_list = [] # Will be used to store dictionaries (from scoring_tool() function) here

    for v in vintages:
        result_dict = scoring_tool(v)
        results_list.append(result_dict)

    # Convert the entire list of results to a DataFrame
    df_production_report = pd.DataFrame(results_list)

    print("\n--- Agent Production Run Complete ---")
    return df_production_report

# Execute
df_master_report = run_ccar_agent( "201605", "201612" )

# Display the final one-liner per vintage
print(df_master_report)


=============== Agent acting on vintage: 201605 ===============

------------- Step B-1: Column List of df_mortgage ---------
 (67957, 44)

------------- Step C-1: Column List of df_mortgage --------:
 report_yrmo
201605    1157
Name: count, dtype: int64
------------- Step C-2: Summary Table of df_mortgage ---------
 flag_default     0   1   All
report_yrmo                 
201605        1131  26  1157
All           1131  26  1157


------- E-1: Classification Report -------
              precision    recall  f1-score   support

           0       0.98      0.94      0.96      1131
           1       0.06      0.19      0.10        26

    accuracy                           0.92      1157
   macro avg       0.52      0.56      0.53      1157
weighted avg       0.96      0.92      0.94      1157

------ E-2 Confusion Matrix ------
[[1058   73]
 [  21    5]]

=============== Agent acting on vintage: 201606 ===============

------------- Step B-1: Column List of df_mortgage ---------
 (6

In [ ]:
from datetime import datetime
end = datetime.now(ZoneInfo("America/New_York"))
duration = end - start

print(f"Started:  {start}")
print(f"Finished: {end}")
print(f"\nDuration: {duration}")                    # 0:00:02.351234
print(f"Duration: {duration.total_seconds():.3f} seconds")

Started:  2026-03-22 22:04:16.235570-04:00
Finished: 2026-03-22 22:04:33.152093-04:00

Duration: 0:00:16.916523
Duration: 16.917 seconds
